# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YassinKhodir/yassin-flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This notebook checks two published findings constructively, then attacks my own Week-5 result with a harder split, a leakage test, and real error examples.

## 1. Two paper findings and my methodology questions

**Finding 1 — “The Freshness Multiplier.”** The paper reports that mature pages refreshed within 30 days had 3.2x higher health and 57x more impressions than an older stale comparison group. My question is: **were refreshed and untouched pages comparable before the refresh on prior visibility, client, content type, and business importance?** Editors may choose their strongest pages for refresh, so an observed difference can reflect selection as well as refresh timing. A matched or staggered before/after comparison would better support a refresh-effect claim.

**Finding 2 — growth logistic regression.** The appendix reports 71% holdout accuracy for separating growing from declining pages. My questions are: **does the holdout keep entire clients or later time periods separate, and are all feature windows strictly before the trend label window?** I would also place 71% beside the class base rate. These checks would show how much of the number transfers beyond the sampled pages.

These are questions about claim scope, not a grade of the paper. The paper itself labels its ML pages exploratory and warns that direct portfolio comparisons lead the evidence.

## 2. My model under an honest split

I compare a normal random page split with a client-grouped split. The random split can place pages from the same client on both sides; the grouped split tests transfer to clients the model never saw. Both use the same five March 1–15 features and March 16–31 label.

In [1]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

required = {
    "duckdb": "duckdb", "pandas": "pandas", "numpy": "numpy",
    "sklearn": "scikit-learn",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import duckdb
import numpy as np
import pandas as pd
import sklearn
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

data_root = os.environ.get("FLYRANK_DATA_ROOT")
if data_root:
    REL = data_root.rstrip("/")
else:
    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token:
        try:
            from google.colab import userdata
            hf_token = userdata.get("HF_TOKEN")
        except Exception:
            hf_token = None
    if not hf_token:
        raise RuntimeError("Add a Hugging Face READ token as a Colab Secret named HF_TOKEN; never paste it into this public notebook.")
    REL = "hf://datasets/FlyRank/internship-warehouse"

con = duckdb.connect()
if not data_root:
    escaped_token = hf_token.replace("'", "''")
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{escaped_token}')")
    del hf_token, escaped_token

MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
feature_query = f"""
WITH early AS (
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS early_impressions,
        SUM(gsc_clicks) AS early_clicks,
        COUNT(*) AS early_observed_days,
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS early_active_days,
        SUM(CASE WHEN gsc_impressions > 0 AND gsc_avg_position IS NOT NULL
                 THEN gsc_avg_position * gsc_impressions ELSE 0 END)
          / NULLIF(SUM(CASE WHEN gsc_impressions > 0 AND gsc_avg_position IS NOT NULL
                            THEN gsc_impressions ELSE 0 END), 0) AS early_avg_position,
        STDDEV_SAMP(CASE WHEN gsc_impressions > 0 THEN gsc_avg_position END) AS early_position_volatility
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100
),
late AS (
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS late_impressions,
           COUNT(*) AS late_observed_days
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
    GROUP BY 1, 2
)
SELECT
    e.client_hash_id, e.content_hash_id,
    LN(1 + e.early_impressions) AS log_early_impressions,
    1.0 * e.early_clicks / NULLIF(e.early_impressions, 0) AS early_ctr,
    e.early_active_days, e.early_avg_position, e.early_position_volatility,
    e.early_impressions,
    CAST((1.0 * l.late_impressions / NULLIF(l.late_observed_days, 0))
         < 0.8 * (1.0 * e.early_impressions / NULLIF(e.early_observed_days, 0)) AS INTEGER)
      AS is_future_decline
FROM early e
JOIN late l USING (client_hash_id, content_hash_id)
ORDER BY e.client_hash_id, e.content_hash_id
"""

frame = con.sql(feature_query).df()
features = [
    "log_early_impressions", "early_ctr", "early_active_days",
    "early_avg_position", "early_position_volatility",
]
print(f"Rows: {len(frame):,}; clients: {frame['client_hash_id'].nunique()}; sklearn: {sklearn.__version__}")

Rows: 77,400; clients: 38; sklearn: 1.8.0


In [2]:
from sklearn.model_selection import train_test_split

def new_model():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(max_iter=1000, random_state=42),
    )

# Before: random page split.
random_train_idx, random_test_idx = train_test_split(
    np.arange(len(frame)), test_size=0.25, random_state=42,
    stratify=frame["is_future_decline"],
)
random_model = new_model()
random_model.fit(frame.iloc[random_train_idx][features], frame.iloc[random_train_idx]["is_future_decline"])
random_prob = random_model.predict_proba(frame.iloc[random_test_idx][features])[:, 1]
random_auc = roc_auc_score(frame.iloc[random_test_idx]["is_future_decline"], random_prob)

# After: hold out entire clients.
group_split = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
group_train_idx, group_test_idx = next(group_split.split(
    frame, frame["is_future_decline"], groups=frame["client_hash_id"]
))
group_train = frame.iloc[group_train_idx].copy()
group_test = frame.iloc[group_test_idx].copy()
assert set(group_train["client_hash_id"]).isdisjoint(set(group_test["client_hash_id"]))

group_model = new_model()
group_model.fit(group_train[features], group_train["is_future_decline"])
group_test["model_score"] = group_model.predict_proba(group_test[features])[:, 1]
group_auc = roc_auc_score(group_test["is_future_decline"], group_test["model_score"])

split_comparison = pd.DataFrame([
    {"split": "random pages", "test_rows": len(random_test_idx),
     "test_clients": frame.iloc[random_test_idx]["client_hash_id"].nunique(),
     "base_rate": frame.iloc[random_test_idx]["is_future_decline"].mean(), "roc_auc": random_auc},
    {"split": "held-out clients", "test_rows": len(group_test),
     "test_clients": group_test["client_hash_id"].nunique(),
     "base_rate": group_test["is_future_decline"].mean(), "roc_auc": group_auc},
]).round(3)
split_comparison

              split  test_rows  test_clients  base_rate  roc_auc
0      random pages      19350            35      0.345    0.594
1  held-out clients      15776            10      0.233    0.510

The score falls from about 0.586 ROC-AUC on randomly mixed pages to about 0.510 on unseen clients. That gap is the main validation finding: much of the apparent random-split signal does not transfer cleanly across clients. The grouped number is the one I keep.

## 3. Leakage audit

Timeline: **March 1–15 features → score after March 15 → March 16–31 observed label.** The five legal features are early impressions, CTR, active days, average position, and position volatility. IDs group rows but are never features. Product flags and every late-window value are excluded.

To test the audit harness, I deliberately copy the label into a feature, verify that grouped ROC-AUC becomes perfect, then delete the column. The perfect score is evidence of leakage, not a model improvement.

In [3]:
legal_features = set(features)
forbidden = {
    "client_hash_id", "content_hash_id", "is_future_decline",
    "late_impressions", "trend_pct", "trend_direction", "product_flag",
}
assert legal_features.isdisjoint(forbidden)

leak_test = frame.copy()
leak_test["leaked_label_copy"] = leak_test["is_future_decline"]
leaky_model = new_model()
leaky_model.fit(
    leak_test.iloc[group_train_idx][features + ["leaked_label_copy"]],
    leak_test.iloc[group_train_idx]["is_future_decline"],
)
leaky_prob = leaky_model.predict_proba(
    leak_test.iloc[group_test_idx][features + ["leaked_label_copy"]]
)[:, 1]
leaky_auc = roc_auc_score(leak_test.iloc[group_test_idx]["is_future_decline"], leaky_prob)
del leak_test["leaked_label_copy"]
assert "leaked_label_copy" not in leak_test.columns

audit = pd.DataFrame([
    {"run": "honest grouped model", "roc_auc": group_auc, "kept": "yes"},
    {"run": "deliberate label leak", "roc_auc": leaky_auc, "kept": "no — deleted"},
]).round(3)
audit

                     run  roc_auc          kept
0   honest grouped model     0.51           yes
1  deliberate label leak     1.00  no — deleted

## 4. Failure examples and claim rewrite

**Old claim (too broad):** “The model predicts which pages will decline and improves the refresh queue.”

**Rewritten claim:** “On one March 2026 held-out-client split, the five-feature logistic score measured ROC-AUC 0.510. It improved precision at the top of the queue compared with the frozen rule, but overall separation was weak. The output is directional decision support for human review, not proof of future decline or refresh impact.”

The displayed error examples omit pseudonymous IDs. They show why search signals alone cannot explain every outcome.

In [4]:
group_test["prediction"] = (group_test["model_score"] >= 0.5).astype(int)
group_test["error_type"] = np.select(
    [(group_test["prediction"] == 1) & (group_test["is_future_decline"] == 0),
     (group_test["prediction"] == 0) & (group_test["is_future_decline"] == 1)],
    ["false positive", "false negative"], default="correct",
)
group_test["confidence"] = np.where(
    group_test["prediction"] == 1, group_test["model_score"], 1 - group_test["model_score"]
)
failure_examples = group_test[group_test["error_type"] != "correct"].nlargest(5, "confidence")[[
    "error_type", "model_score", "log_early_impressions", "early_ctr",
    "early_active_days", "early_avg_position", "early_position_volatility",
]].round(3)
failure_examples

           error_type  ...  early_position_volatility
22910  false negative  ...                      3.306
22906  false negative  ...                      0.464
64662  false negative  ...                      1.805
58921  false negative  ...                      1.143
22922  false negative  ...                      3.768

[5 rows x 7 columns]

In [5]:
def baseline_score(data):
    volume = pd.cut(data["early_impressions"], [-1, 299, 999, 2999, 9999, np.inf], labels=[0, 1, 2, 3, 4]).astype(int)
    position = np.select([
        data["early_avg_position"].between(20, 50, inclusive="right"),
        data["early_avg_position"].between(10, 20, inclusive="right"),
        data["early_avg_position"].between(3, 10, inclusive="right"),
    ], [3, 2, 1], default=0)
    return volume + position

group_test["baseline_score"] = baseline_score(group_test)
precision_model_100 = group_test.nlargest(100, "model_score")["is_future_decline"].mean()
baseline_ranked = group_test.sort_values(
    ["baseline_score", "early_impressions", "early_avg_position", "content_hash_id"],
    ascending=[False, False, True, True],
)
precision_baseline_100 = baseline_ranked.head(100)["is_future_decline"].mean()

output_dir = Path("../outputs") if Path.cwd().name == "notebooks" else Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)
metrics = {
    "random_split_roc_auc": round(float(random_auc), 6),
    "grouped_split_roc_auc": round(float(group_auc), 6),
    "deliberate_leak_roc_auc": round(float(leaky_auc), 6),
    "grouped_test_base_rate": round(float(group_test["is_future_decline"].mean()), 6),
    "grouped_baseline_precision_at_100": round(float(precision_baseline_100), 6),
    "grouped_model_precision_at_100": round(float(precision_model_100), 6),
    "grouped_test_clients": int(group_test["client_hash_id"].nunique()),
}
with open(output_dir / "ml09_validation_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(pd.Series(metrics).to_string())

random_split_roc_auc                  0.593642
grouped_split_roc_auc                 0.509776
deliberate_leak_roc_auc               1.000000
grouped_test_base_rate                0.233266
grouped_baseline_precision_at_100     0.150000
grouped_model_precision_at_100        0.310000
grouped_test_clients                 10.000000


## Self-check

- [x] Two paper findings have specific, constructive methodology questions
- [x] Random-page and held-out-client results are compared
- [x] Feature and label windows do not overlap
- [x] IDs and product flags are excluded from features
- [x] A deliberate leak is detected, removed, and not kept
- [x] Real failure examples omit IDs
- [x] The final claim uses measured, directional, decision-support language
- [x] Aggregate validation metrics are saved as a reproducibility receipt
- [x] The notebook runs top to bottom without errors